# 🎮 Tutorial: Construyendo un Servidor MCP para Pokémon

## Introducción al Model Context Protocol (MCP)

Este notebook te guiará paso a paso a través del proceso completo de construcción de un servidor MCP (Model Context Protocol) para gestionar datos de Pokémon. Aprenderás:

- **¿Qué es MCP?**: El Model Context Protocol es un protocolo estándar que permite a las aplicaciones de IA acceder a herramientas y recursos externos
- **Arquitectura del servidor**: Cómo estructurar un servidor MCP usando Python y Pydantic
- **Implementación de herramientas**: Creación de 3 herramientas específicas para datos de Pokémon
- **Protocolo JSON-RPC**: Comunicación mediante JSON-RPC 2.0 sobre stdin/stdout

### 📋 Objetivos de Aprendizaje

Al final de este tutorial serás capaz de:
1. Entender la estructura básica de un servidor MCP
2. Implementar modelos de datos usando Pydantic
3. Crear herramientas que respondan a solicitudes específicas
4. Manejar comunicación JSON-RPC de forma robusta
5. Desarrollar tu propio servidor MCP para cualquier dominio de datos

### 🛠️ Herramientas que Implementaremos

1. **`get_all_pokemon`**: Obtener todos los Pokémon del almacén de datos
2. **`get_pokemon_by_type`**: Filtrar Pokémon por tipo (Eléctrico, Fuego, Agua, etc.)
3. **`get_pokemon_details`**: Obtener información detallada de un Pokémon específico

¡Comencemos!

## 1. Setup e Importación de Librerías

Primero, importamos todas las librerías necesarias para construir nuestro servidor MCP. Cada una tiene un propósito específico:

- **`asyncio`**: Para manejar operaciones asíncronas
- **`json`**: Para serialización JSON-RPC
- **`sys`**: Para comunicación stdin/stdout
- **`typing`**: Para anotaciones de tipos
- **`pydantic`**: Para validación y modelado de datos

In [ ]:
#!/usr/bin/env python3
"""
Pokemon MCP Server Tutorial
Implementación paso a paso de un servidor MCP para gestión de datos de Pokémon
"""

import asyncio
import json
import sys
from typing import Any, Dict, List, Optional, Union
from pydantic import BaseModel

print("✅ Librerías importadas correctamente")
print("📦 Versiones:")
print(f"   - Python: {sys.version}")
print(f"   - Pydantic: Disponible para modelado de datos")
print(f"   - asyncio: Disponible para operaciones asíncronas")

## 2. Definir el Modelo de Datos de Pokémon

Usamos **Pydantic** para crear un modelo de datos robusto que define la estructura de cada Pokémon. Esto nos proporciona:

- **Validación automática** de tipos de datos
- **Serialización/deserialización** JSON automática
- **Documentación clara** de la estructura de datos
- **Type hints** para mejor experiencia de desarrollo

### Campos del Modelo Pokémon:
- `id`: Identificador único (entero)
- `name`: Nombre del Pokémon (string)
- `type`: Tipo elemental (string)
- `weight`: Peso en kilogramos (float)
- `height`: Altura en metros (float)
- `image_url`: URL de la imagen del sprite (string)

In [ ]:
# Modelo de datos para Pokémon usando Pydantic
class Pokemon(BaseModel):
    id: int
    name: str
    type: str
    weight: float  # en kg
    height: float  # en metros
    image_url: str

# Ejemplo de uso del modelo
pikachu_example = Pokemon(
    id=1,
    name="Pikachu",
    type="Electric",
    weight=6.0,
    height=0.4,
    image_url="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/25.png"
)

print("✅ Modelo Pokemon definido correctamente")
print("📋 Ejemplo de Pokémon:")
print(f"   - Nombre: {pikachu_example.name}")
print(f"   - Tipo: {pikachu_example.type}")
print(f"   - Peso: {pikachu_example.weight} kg")
print(f"   - Altura: {pikachu_example.height} m")
print(f"   - ID: {pikachu_example.id}")

# Demostrar serialización JSON
print("\n🔄 Serialización JSON:")
print(pikachu_example.model_dump())

## 3. Crear Dataset de Pokémon de Prueba

Ahora creamos un almacén de datos en memoria con 10 Pokémon populares. Este dataset incluye variedad de tipos y estadísticas realistas.

### Pokémon Incluidos:
1. **Pikachu** (Eléctrico) - La mascota de Pokémon
2. **Charizard** (Fuego) - Dragón de fuego icónico
3. **Blastoise** (Agua) - Tortuga con cañones de agua
4. **Venusaur** (Planta) - Dinosaurio con flor
5. **Gyarados** (Agua) - Serpiente marina gigante
6. **Alakazam** (Psíquico) - Mago con cucharas
7. **Machamp** (Lucha) - Luchador de cuatro brazos
8. **Gengar** (Fantasma) - Pokémon fantasma travieso
9. **Dragonite** (Dragón) - Dragón amigable
10. **Mewtwo** (Psíquico) - Pokémon legendario creado genéticamente

In [ ]:
# Dataset de Pokémon para nuestro servidor MCP
POKEMON_DATA: List[Pokemon] = [
    Pokemon(
        id=1, name="Pikachu", type="Electric", weight=6.0, height=0.4,
        image_url="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/25.png"
    ),
    Pokemon(
        id=2, name="Charizard", type="Fire", weight=90.5, height=1.7,
        image_url="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/6.png"
    ),
    Pokemon(
        id=3, name="Blastoise", type="Water", weight=85.5, height=1.6,
        image_url="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/9.png"
    ),
    Pokemon(
        id=4, name="Venusaur", type="Grass", weight=100.0, height=2.0,
        image_url="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/3.png"
    ),
    Pokemon(
        id=5, name="Gyarados", type="Water", weight=235.0, height=6.5,
        image_url="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/130.png"
    ),
    Pokemon(
        id=6, name="Alakazam", type="Psychic", weight=48.0, height=1.5,
        image_url="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/65.png"
    ),
    Pokemon(
        id=7, name="Machamp", type="Fighting", weight=130.0, height=1.6,
        image_url="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/68.png"
    ),
    Pokemon(
        id=8, name="Gengar", type="Ghost", weight=40.5, height=1.5,
        image_url="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/94.png"
    ),
    Pokemon(
        id=9, name="Dragonite", type="Dragon", weight=210.0, height=2.2,
        image_url="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/149.png"
    ),
    Pokemon(
        id=10, name="Mewtwo", type="Psychic", weight=122.0, height=2.0,
        image_url="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/150.png"
    )
]

print(f"✅ Dataset creado con {len(POKEMON_DATA)} Pokémon")
print("\n📊 Estadísticas del dataset:")
print(f"   - Total de Pokémon: {len(POKEMON_DATA)}")

# Análisis de tipos
types_count = {}
for pokemon in POKEMON_DATA:
    types_count[pokemon.type] = types_count.get(pokemon.type, 0) + 1

print("   - Distribución por tipos:")
for ptype, count in types_count.items():
    print(f"     • {ptype}: {count} Pokémon")

# Peso promedio
avg_weight = sum(p.weight for p in POKEMON_DATA) / len(POKEMON_DATA)
print(f"   - Peso promedio: {avg_weight:.1f} kg")

# Altura promedio
avg_height = sum(p.height for p in POKEMON_DATA) / len(POKEMON_DATA)
print(f"   - Altura promedio: {avg_height:.1f} m")

## 4. Implementar las Clases del Protocolo MCP

El Model Context Protocol se basa en **JSON-RPC 2.0** para la comunicación. Necesitamos definir modelos para:

### MCPRequest (Solicitud)
- `jsonrpc`: Versión del protocolo (siempre "2.0")
- `id`: Identificador único de la solicitud (opcional)
- `method`: Método a ejecutar (ej: "initialize", "tools/list", "tools/call")
- `params`: Parámetros específicos del método (opcional)

### MCPResponse (Respuesta)
- `jsonrpc`: Versión del protocolo (siempre "2.0")
- `id`: Mismo ID de la solicitud correspondiente
- `result`: Datos de respuesta exitosa (opcional)
- `error`: Información de error si algo falló (opcional)

### Flujo de Comunicación MCP:
1. **Cliente** envía `MCPRequest` con un método
2. **Servidor** procesa la solicitud
3. **Servidor** responde con `MCPResponse` conteniendo resultado o error

In [ ]:
# Modelos para el protocolo MCP (JSON-RPC 2.0)
class MCPRequest(BaseModel):
    jsonrpc: str = "2.0"
    id: Optional[Union[str, int]] = None
    method: str
    params: Optional[Dict[str, Any]] = None

class MCPResponse(BaseModel):
    jsonrpc: str = "2.0"
    id: Optional[Union[str, int]] = None
    result: Optional[Any] = None
    error: Optional[Dict[str, Any]] = None

# Ejemplos de uso de los modelos MCP
print("✅ Modelos MCP definidos correctamente")

# Ejemplo de solicitud
example_request = MCPRequest(
    id=1,
    method="tools/list"
)
print("\n📤 Ejemplo de MCPRequest:")
print(json.dumps(example_request.model_dump(exclude_none=True), indent=2))

# Ejemplo de respuesta exitosa
example_success_response = MCPResponse(
    id=1,
    result={"tools": [{"name": "get_all_pokemon", "description": "Obtener todos los Pokémon"}]}
)
print("\n📥 Ejemplo de MCPResponse exitosa:")
print(json.dumps(example_success_response.model_dump(exclude_none=True), indent=2))

# Ejemplo de respuesta con error
example_error_response = MCPResponse(
    id=1,
    error={"code": -32601, "message": "Método no encontrado"}
)
print("\n❌ Ejemplo de MCPResponse con error:")
print(json.dumps(example_error_response.model_dump(exclude_none=True), indent=2))

## 5. Construir la Clase Principal del Servidor MCP

La clase `MCPServer` es el corazón de nuestro servidor. Maneja:

### Inicialización y Capacidades
- Define las **capacidades** del servidor (qué puede hacer)
- Responde a solicitudes de **inicialización** del cliente
- Proporciona información del servidor (nombre, versión)

### Métodos Principales que Maneja:
1. **`initialize`**: Establece la conexión y negocia capacidades
2. **`tools/list`**: Lista todas las herramientas disponibles
3. **`tools/call`**: Ejecuta una herramienta específica

### Manejo de Errores:
- **-32601**: Método no encontrado
- **-32602**: Parámetros inválidos
- **-32603**: Error interno del servidor
- **-32700**: Error de parsing JSON

In [ ]:
class MCPServer:
    def __init__(self):
        # Definir las capacidades del servidor
        self.capabilities = {
            "tools": {
                "listChanged": True  # El servidor puede notificar cambios en las herramientas
            }
        }
        
    async def handle_request(self, request: MCPRequest) -> MCPResponse:
        """Manejador principal de solicitudes MCP"""
        try:
            if request.method == "initialize":
                # Manejar inicialización del servidor
                return MCPResponse(
                    id=request.id,
                    result={
                        "protocolVersion": "2024-11-05",
                        "capabilities": self.capabilities,
                        "serverInfo": {
                            "name": "pokemon-server",
                            "version": "1.0.0"
                        }
                    }
                )
            
            elif request.method == "tools/list":
                # Listar todas las herramientas disponibles
                return MCPResponse(
                    id=request.id,
                    result={
                        "tools": [
                            {
                                "name": "get_all_pokemon",
                                "description": "Obtener todos los Pokémon del almacén de datos",
                                "inputSchema": {
                                    "type": "object",
                                    "properties": {},
                                    "required": []
                                }
                            },
                            {
                                "name": "get_pokemon_by_type",
                                "description": "Obtener Pokémon filtrados por tipo",
                                "inputSchema": {
                                    "type": "object",
                                    "properties": {
                                        "pokemon_type": {
                                            "type": "string",
                                            "description": "Tipo de Pokémon a filtrar (ej: Electric, Fire, Water)"
                                        }
                                    },
                                    "required": ["pokemon_type"]
                                }
                            },
                            {
                                "name": "get_pokemon_details",
                                "description": "Obtener información detallada de un Pokémon específico",
                                "inputSchema": {
                                    "type": "object",
                                    "properties": {
                                        "identifier": {
                                            "type": "string",
                                            "description": "Nombre o ID del Pokémon"
                                        }
                                    },
                                    "required": ["identifier"]
                                }
                            }
                        ]
                    }
                )
            
            elif request.method == "tools/call":
                # Ejecutar una herramienta específica (implementaremos esto en la siguiente sección)
                return await self.handle_tool_call(request)
            
            else:
                # Método no encontrado
                return MCPResponse(
                    id=request.id,
                    error={
                        "code": -32601,
                        "message": f"Método no encontrado: {request.method}"
                    }
                )
                
        except Exception as e:
            # Error interno del servidor
            return MCPResponse(
                id=request.id,
                error={
                    "code": -32603,
                    "message": f"Error interno: {str(e)}"
                }
            )
    
    async def handle_tool_call(self, request: MCPRequest) -> MCPResponse:
        """Placeholder para el manejador de herramientas (implementaremos en la siguiente sección)"""
        return MCPResponse(
            id=request.id,
            error={
                "code": -32603,
                "message": "handle_tool_call no implementado aún"
            }
        )

# Crear instancia del servidor para pruebas
server = MCPServer()
print("✅ Servidor MCP creado correctamente")
print(f"🔧 Capacidades del servidor: {server.capabilities}")

## 6. Implementar el Manejador de Herramientas

El método `handle_tool_call` es donde implementamos la lógica de negocio de nuestras herramientas. Cada herramienta tiene una responsabilidad específica:

### 🔍 get_all_pokemon
- **Propósito**: Devolver la lista completa de Pokémon
- **Parámetros**: Ninguno
- **Retorna**: Array con todos los Pokémon y conteo total

### 🏷️ get_pokemon_by_type
- **Propósito**: Filtrar Pokémon por tipo elemental
- **Parámetros**: `pokemon_type` (string)
- **Retorna**: Array de Pokémon del tipo especificado
- **Manejo de errores**: Si no encuentra el tipo, lista los tipos disponibles

### 📋 get_pokemon_details
- **Propósito**: Información detallada de un Pokémon específico
- **Parámetros**: `identifier` (nombre o ID)
- **Retorna**: Datos completos + cálculos adicionales (IMC)
- **Búsqueda inteligente**: Primero por ID, luego por nombre (case-insensitive)

In [ ]:
# Implementación completa del manejador de herramientas
async def handle_tool_call_implementation(request: MCPRequest) -> MCPResponse:
    """Implementación completa del manejador de llamadas a herramientas"""
    if not request.params:
        return MCPResponse(
            id=request.id,
            error={
                "code": -32602,
                "message": "Parámetros inválidos"
            }
        )
    
    tool_name = request.params.get("name")
    arguments = request.params.get("arguments", {})
    
    if tool_name == "get_all_pokemon":
        # 🔍 Herramienta 1: Obtener todos los Pokémon
        pokemon_list = []
        for pokemon in POKEMON_DATA:
            pokemon_list.append({
                "id": pokemon.id,
                "name": pokemon.name,
                "type": pokemon.type,
                "weight": pokemon.weight,
                "height": pokemon.height,
                "image_url": pokemon.image_url
            })
        
        result = {
            "total_count": len(pokemon_list),
            "pokemon": pokemon_list
        }
        
        return MCPResponse(
            id=request.id,
            result={
                "content": [
                    {
                        "type": "text",
                        "text": f"Encontrados {len(pokemon_list)} Pokémon:\\n\\n{json.dumps(result, indent=2)}"
                    }
                ]
            }
        )
    
    elif tool_name == "get_pokemon_by_type":
        # 🏷️ Herramienta 2: Filtrar por tipo
        pokemon_type = arguments.get("pokemon_type", "").title()
        
        filtered_pokemon = [
            pokemon for pokemon in POKEMON_DATA 
            if pokemon.type.lower() == pokemon_type.lower()
        ]
        
        if not filtered_pokemon:
            available_types = list(set(p.type for p in POKEMON_DATA))
            return MCPResponse(
                id=request.id,
                result={
                    "content": [
                        {
                            "type": "text",
                            "text": f"No se encontraron Pokémon de tipo '{pokemon_type}'. Tipos disponibles: {', '.join(available_types)}"
                        }
                    ]
                }
            )
        
        pokemon_list = []
        for pokemon in filtered_pokemon:
            pokemon_list.append({
                "id": pokemon.id,
                "name": pokemon.name,
                "type": pokemon.type,
                "weight": pokemon.weight,
                "height": pokemon.height,
                "image_url": pokemon.image_url
            })
        
        result = {
            "type_filter": pokemon_type,
            "count": len(pokemon_list),
            "pokemon": pokemon_list
        }
        
        return MCPResponse(
            id=request.id,
            result={
                "content": [
                    {
                        "type": "text",
                        "text": f"Encontrados {len(pokemon_list)} Pokémon de tipo '{pokemon_type}':\\n\\n{json.dumps(result, indent=2)}"
                    }
                ]
            }
        )
    
    elif tool_name == "get_pokemon_details":
        # 📋 Herramienta 3: Detalles específicos
        identifier = arguments.get("identifier", "").strip()
        
        found_pokemon = None
        
        # Intentar buscar por ID primero
        try:
            pokemon_id = int(identifier)
            found_pokemon = next((p for p in POKEMON_DATA if p.id == pokemon_id), None)
        except ValueError:
            # No es un número, buscar por nombre
            found_pokemon = next(
                (p for p in POKEMON_DATA if p.name.lower() == identifier.lower()), 
                None
            )
        
        if not found_pokemon:
            available_pokemon = [f"{p.name} (ID: {p.id})" for p in POKEMON_DATA]
            return MCPResponse(
                id=request.id,
                result={
                    "content": [
                        {
                            "type": "text",
                            "text": f"Pokémon '{identifier}' no encontrado. Pokémon disponibles:\\n{chr(10).join(available_pokemon)}"
                        }
                    ]
                }
            )
        
        # Calcular información adicional
        bmi = round(found_pokemon.weight / (found_pokemon.height ** 2), 2)
        
        result = {
            "id": found_pokemon.id,
            "name": found_pokemon.name,
            "type": found_pokemon.type,
            "weight": found_pokemon.weight,
            "height": found_pokemon.height,
            "image_url": found_pokemon.image_url,
            "details": {
                "weight_description": f"{found_pokemon.weight} kg",
                "height_description": f"{found_pokemon.height} m",
                "bmi": bmi
            }
        }
        
        return MCPResponse(
            id=request.id,
            result={
                "content": [
                    {
                        "type": "text",
                        "text": f"Detalles de {found_pokemon.name}:\\n\\n{json.dumps(result, indent=2)}"
                    }
                ]
            }
        )
    
    else:
        return MCPResponse(
            id=request.id,
            error={
                "code": -32601,
                "message": f"Herramienta desconocida: {tool_name}"
            }
        )

# Actualizar la clase MCPServer con la implementación completa
MCPServer.handle_tool_call = handle_tool_call_implementation

print("✅ Manejador de herramientas implementado correctamente")
print("🛠️ Herramientas disponibles:")
print("   1. get_all_pokemon - Obtener todos los Pokémon")
print("   2. get_pokemon_by_type - Filtrar por tipo")
print("   3. get_pokemon_details - Detalles específicos")

## 7. Probar las Herramientas Individualmente

Ahora vamos a probar cada herramienta por separado para asegurar que funcionen correctamente antes de integrarlas en el servidor completo.

### Plan de Pruebas:
1. **✅ Inicialización**: Verificar que el servidor responde correctamente
2. **📋 Listar herramientas**: Confirmar que las 3 herramientas están disponibles
3. **🔍 get_all_pokemon**: Obtener la lista completa
4. **🏷️ get_pokemon_by_type**: Filtrar por tipo "Fire" y tipo inválido
5. **📋 get_pokemon_details**: Buscar por nombre y por ID

Estas pruebas nos ayudan a identificar problemas antes de la implementación final.

In [ ]:
# Función helper para ejecutar pruebas asíncronas
async def test_server_functionality():
    """Prueba todas las funcionalidades del servidor MCP"""
    server = MCPServer()
    print("🧪 Iniciando pruebas del servidor MCP")
    print("=" * 50)
    
    # Prueba 1: Inicialización
    print("\\n1️⃣ Prueba de Inicialización")
    init_request = MCPRequest(id=1, method="initialize")
    response = await server.handle_request(init_request)
    
    if response.result:
        print(f"✅ Inicialización exitosa")
        print(f"   Servidor: {response.result['serverInfo']['name']}")
        print(f"   Versión: {response.result['serverInfo']['version']}")
    else:
        print(f"❌ Error en inicialización: {response.error}")
    
    # Prueba 2: Listar herramientas
    print("\\n2️⃣ Prueba de Listado de Herramientas")
    tools_request = MCPRequest(id=2, method="tools/list")
    response = await server.handle_request(tools_request)
    
    if response.result:
        tools = response.result["tools"]
        print(f"✅ {len(tools)} herramientas encontradas:")
        for i, tool in enumerate(tools, 1):
            print(f"   {i}. {tool['name']}: {tool['description']}")
    else:
        print(f"❌ Error listando herramientas: {response.error}")
    
    # Prueba 3: get_all_pokemon
    print("\\n3️⃣ Prueba: Obtener todos los Pokémon")
    all_request = MCPRequest(
        id=3,
        method="tools/call",
        params={"name": "get_all_pokemon", "arguments": {}}
    )
    response = await server.handle_request(all_request)
    
    if response.result:
        content = response.result["content"][0]["text"]
        lines = content.split('\\n')
        print(f"✅ {lines[0]}")
        # Extraer información básica del JSON
        try:
            json_start = content.find('{')
            data = json.loads(content[json_start:])
            print(f"   Total: {data['total_count']} Pokémon")
            print(f"   Primeros 3: {', '.join(p['name'] for p in data['pokemon'][:3])}")
        except:
            print("   (No se pudo parsear detalles)")
    else:
        print(f"❌ Error: {response.error}")
    
    # Prueba 4: get_pokemon_by_type (tipo válido)
    print("\\n4️⃣ Prueba: Filtrar por tipo 'Fire'")
    fire_request = MCPRequest(
        id=4,
        method="tools/call",
        params={"name": "get_pokemon_by_type", "arguments": {"pokemon_type": "Fire"}}
    )
    response = await server.handle_request(fire_request)
    
    if response.result:
        content = response.result["content"][0]["text"]
        lines = content.split('\\n')
        print(f"✅ {lines[0]}")
        try:
            json_start = content.find('{')
            data = json.loads(content[json_start:])
            pokemon_names = [p['name'] for p in data['pokemon']]
            print(f"   Pokémon Fire: {', '.join(pokemon_names)}")
        except:
            print("   (Resultado sin formato JSON)")
    else:
        print(f"❌ Error: {response.error}")
    
    # Prueba 5: get_pokemon_by_type (tipo inválido)
    print("\\n5️⃣ Prueba: Filtrar por tipo inválido 'Ice'")
    ice_request = MCPRequest(
        id=5,
        method="tools/call",
        params={"name": "get_pokemon_by_type", "arguments": {"pokemon_type": "Ice"}}
    )
    response = await server.handle_request(ice_request)
    
    if response.result:
        content = response.result["content"][0]["text"]
        print(f"✅ Manejo correcto de tipo inválido")
        print(f"   Mensaje: {content.split('.')[0]}.")
    else:
        print(f"❌ Error: {response.error}")
    
    # Prueba 6: get_pokemon_details (búsqueda por nombre)
    print("\\n6️⃣ Prueba: Detalles de Pikachu (por nombre)")
    pikachu_request = MCPRequest(
        id=6,
        method="tools/call",
        params={"name": "get_pokemon_details", "arguments": {"identifier": "Pikachu"}}
    )
    response = await server.handle_request(pikachu_request)
    
    if response.result:
        content = response.result["content"][0]["text"]
        lines = content.split('\\n')
        print(f"✅ {lines[0]}")
        try:
            json_start = content.find('{')
            data = json.loads(content[json_start:])
            print(f"   Tipo: {data['type']}")
            print(f"   Peso: {data['weight']} kg")
            print(f"   IMC: {data['details']['bmi']}")
        except:
            print("   (No se pudo parsear detalles)")
    else:
        print(f"❌ Error: {response.error}")
    
    # Prueba 7: get_pokemon_details (búsqueda por ID)
    print("\\n7️⃣ Prueba: Detalles del Pokémon ID 2")
    id_request = MCPRequest(
        id=7,
        method="tools/call",
        params={"name": "get_pokemon_details", "arguments": {"identifier": "2"}}
    )
    response = await server.handle_request(id_request)
    
    if response.result:
        content = response.result["content"][0]["text"]
        lines = content.split('\\n')
        print(f"✅ {lines[0]}")
        try:
            json_start = content.find('{')
            data = json.loads(content[json_start:])
            print(f"   Nombre: {data['name']}")
            print(f"   Tipo: {data['type']}")
        except:
            print("   (No se pudo parsear detalles)")
    else:
        print(f"❌ Error: {response.error}")
    
    print("\\n🎉 Todas las pruebas completadas!")

# Ejecutar las pruebas
await test_server_functionality()

## 8. Manejar Comunicación JSON-RPC

El protocolo MCP usa **JSON-RPC 2.0** para la comunicación entre cliente y servidor. Aquí implementamos:

### Parsing de Solicitudes
- Leer líneas JSON desde `stdin`
- Validar formato JSON-RPC
- Convertir a objetos `MCPRequest`
- Manejar errores de parsing

### Formateo de Respuestas
- Convertir `MCPResponse` a JSON
- Enviar respuestas a `stdout`
- Flush para asegurar entrega inmediata

### Códigos de Error JSON-RPC:
- **-32700**: Parse error (JSON malformado)
- **-32600**: Invalid Request (estructura incorrecta)
- **-32601**: Method not found (método no existe)
- **-32602**: Invalid params (parámetros incorrectos)
- **-32603**: Internal error (error del servidor)

In [ ]:
# Simulación de comunicación JSON-RPC
def simulate_json_rpc_communication():
    """Simula el procesamiento de comunicación JSON-RPC"""
    
    # Ejemplos de mensajes JSON-RPC que el servidor podría recibir
    sample_messages = [
        # Mensaje válido de inicialización
        {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "initialize",
            "params": {
                "protocolVersion": "2024-11-05",
                "capabilities": {},
                "clientInfo": {"name": "test-client", "version": "1.0.0"}
            }
        },
        
        # Mensaje válido para listar herramientas
        {
            "jsonrpc": "2.0",
            "id": 2,
            "method": "tools/list"
        },
        
        # Mensaje válido para llamar herramienta
        {
            "jsonrpc": "2.0",
            "id": 3,
            "method": "tools/call",
            "params": {
                "name": "get_pokemon_by_type",
                "arguments": {"pokemon_type": "Electric"}
            }
        },
        
        # Mensaje con JSON malformado (simulado como string)
        "{ invalid json }",
        
        # Mensaje con método inexistente
        {
            "jsonrpc": "2.0",
            "id": 4,
            "method": "nonexistent_method"
        }
    ]
    
    print("🔄 Simulación de Comunicación JSON-RPC")
    print("=" * 50)
    
    for i, message in enumerate(sample_messages, 1):
        print(f"\\n📤 Mensaje {i}:")
        
        if isinstance(message, str):
            # Simular JSON malformado
            print(f"   JSON: {message}")
            print("❌ Error de parsing: JSON malformado")
            
            error_response = MCPResponse(
                error={
                    "code": -32700,
                    "message": "Parse error: JSON malformado"
                }
            )
            print(f"📥 Respuesta: {json.dumps(error_response.model_dump(exclude_none=True))}")
            
        else:
            # Procesar mensaje válido
            try:
                print(f"   JSON: {json.dumps(message, indent=6)}")
                
                # Convertir a MCPRequest
                request = MCPRequest(**message)
                print(f"✅ Parsing exitoso: método '{request.method}'")
                
                # Esta sería la respuesta del servidor (simplificada)
                if request.method == "initialize":
                    response = MCPResponse(
                        id=request.id,
                        result={"protocolVersion": "2024-11-05", "serverInfo": {"name": "pokemon-server"}}
                    )
                    print(f"📥 Respuesta: Inicialización exitosa")
                    
                elif request.method == "tools/list":
                    response = MCPResponse(
                        id=request.id,
                        result={"tools": [{"name": "get_all_pokemon"}, {"name": "get_pokemon_by_type"}]}
                    )
                    print(f"📥 Respuesta: Lista de herramientas")
                    
                elif request.method == "tools/call":
                    response = MCPResponse(
                        id=request.id,
                        result={"content": [{"type": "text", "text": "Resultado de herramienta"}]}
                    )
                    print(f"📥 Respuesta: Resultado de herramienta")
                    
                else:
                    response = MCPResponse(
                        id=request.id,
                        error={"code": -32601, "message": f"Method not found: {request.method}"}
                    )
                    print(f"❌ Respuesta: Método no encontrado")
                
            except Exception as e:
                print(f"❌ Error procesando mensaje: {e}")
                response = MCPResponse(
                    error={"code": -32600, "message": "Invalid Request"}
                )
                print(f"📥 Respuesta: Solicitud inválida")

# Demostrar manejo de errores JSON-RPC
def demonstrate_error_handling():
    """Demuestra diferentes tipos de errores JSON-RPC"""
    
    print("\\n🚨 Demostración de Manejo de Errores")
    print("=" * 40)
    
    error_examples = [
        {
            "code": -32700,
            "message": "Parse error",
            "description": "JSON malformado o no válido"
        },
        {
            "code": -32600,
            "message": "Invalid Request",
            "description": "Estructura de solicitud incorrecta"
        },
        {
            "code": -32601,
            "message": "Method not found",
            "description": "El método solicitado no existe"
        },
        {
            "code": -32602,
            "message": "Invalid params",
            "description": "Parámetros inválidos o faltantes"
        },
        {
            "code": -32603,
            "message": "Internal error",
            "description": "Error interno del servidor"
        }
    ]
    
    for error in error_examples:
        print(f"\\n❌ Error {error['code']}: {error['message']}")
        print(f"   📝 {error['description']}")
        
        # Crear respuesta de error
        error_response = MCPResponse(
            id=1,
            error={
                "code": error["code"],
                "message": error["message"]
            }
        )
        print(f"   🔄 JSON: {json.dumps(error_response.model_dump(exclude_none=True))}")

# Ejecutar simulaciones
simulate_json_rpc_communication()
demonstrate_error_handling()

## 9. Crear el Bucle Principal del Servidor

El bucle principal es donde el servidor MCP "vive" y procesa solicitudes de forma continua. Implementamos:

### Ciclo de Comunicación:
1. **Leer** de `stdin` (solicitudes JSON-RPC del cliente)
2. **Parsear** JSON y validar estructura
3. **Procesar** solicitud usando nuestros manejadores
4. **Responder** escribiendo JSON a `stdout`
5. **Repetir** hasta recibir interrupción

### Características Importantes:
- **Asíncrono**: Usa `asyncio` para manejar operaciones no bloqueantes
- **Robusto**: Maneja errores sin crash del servidor
- **Flush**: Asegura que las respuestas se envíen inmediatamente
- **Limpieza**: Manejo de `KeyboardInterrupt` para cierre limpio

### Flujo de Datos:
```
Cliente → stdin → JSON Parse → MCPRequest → handle_request → MCPResponse → JSON → stdout → Cliente
```

In [ ]:
# Implementación del bucle principal del servidor MCP
async def main_server_loop():
    """
    Bucle principal del servidor MCP que maneja comunicación stdin/stdout
    Nota: Esta es una versión educativa - en producción se ejecutaría indefinidamente
    """
    server = MCPServer()
    
    print("🚀 Servidor Pokemon MCP iniciado")
    print("👂 Escuchando solicitudes JSON-RPC...")
    print("💡 En producción, este bucle correría indefinidamente")
    print("\\n" + "="*50)
    
    # En un servidor real, esto sería un bucle infinito leyendo de stdin
    # Aquí simulamos con algunas solicitudes de ejemplo
    
    # Simular solicitudes que llegarían por stdin
    sample_requests = [
        '{"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2024-11-05"}}',
        '{"jsonrpc": "2.0", "id": 2, "method": "tools/list"}',
        '{"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "get_all_pokemon", "arguments": {}}}',
        '{"jsonrpc": "2.0", "id": 4, "method": "tools/call", "params": {"name": "get_pokemon_by_type", "arguments": {"pokemon_type": "Electric"}}}',
        '{"jsonrpc": "2.0", "id": 5, "method": "tools/call", "params": {"name": "get_pokemon_details", "arguments": {"identifier": "Charizard"}}}'
    ]
    
    for line_num, line in enumerate(sample_requests, 1):
        print(f"\\n📥 Solicitud {line_num} recibida:")
        print(f"   {line[:80]}{'...' if len(line) > 80 else ''}")
        
        try:
            # Parsear JSON-RPC request
            try:
                request_data = json.loads(line)
                request = MCPRequest(**request_data)
                print(f"✅ Parsing exitoso: {request.method}")
            except (json.JSONDecodeError, ValueError) as e:
                print(f"❌ Error de parsing: {str(e)}")
                error_response = MCPResponse(
                    error={
                        "code": -32700,
                        "message": f"Parse error: {str(e)}"
                    }
                )
                response_json = json.dumps(error_response.model_dump(exclude_none=True))
                print(f"📤 Respuesta de error enviada: {response_json[:100]}...")
                continue
            
            # Manejar solicitud
            response = await server.handle_request(request)
            
            # Enviar respuesta (en producción iría a stdout)
            response_json = json.dumps(response.model_dump(exclude_none=True))
            print(f"📤 Respuesta enviada:")
            
            # Mostrar respuesta formateada (solo para fines educativos)
            if len(response_json) > 200:
                print(f"   {response_json[:200]}...")
                print(f"   [Respuesta completa: {len(response_json)} caracteres]")
            else:
                formatted_response = json.dumps(response.model_dump(exclude_none=True), indent=2)
                print(f"   {formatted_response}")
            
            # En producción: sys.stdout.flush() para asegurar entrega inmediata
            
        except KeyboardInterrupt:
            print("\\n⚠️ Interrupción recibida - cerrando servidor...")
            break
        except Exception as e:
            print(f"❌ Error inesperado: {e}")
            break
    
    print("\\n🛑 Servidor detenido")
    return "Simulación completada"

# Función que muestra el código del bucle principal real
def show_production_main_loop():
    """Muestra cómo sería el bucle principal en producción"""
    
    production_code = '''
async def main():
    """Bucle principal del servidor MCP en producción"""
    server = MCPServer()
    
    print("Pokemon MCP Server started. Listening for requests...", file=sys.stderr)
    
    while True:
        try:
            # Leer solicitud de stdin (bloqueante)
            line = await asyncio.get_event_loop().run_in_executor(None, sys.stdin.readline)
            if not line:
                break
            
            line = line.strip()
            if not line:
                continue
            
            # Parsear JSON-RPC request
            try:
                request_data = json.loads(line)
                request = MCPRequest(**request_data)
            except (json.JSONDecodeError, ValueError) as e:
                error_response = MCPResponse(
                    error={
                        "code": -32700,
                        "message": f"Parse error: {str(e)}"
                    }
                )
                print(json.dumps(error_response.model_dump(exclude_none=True)))
                continue
            
            # Manejar solicitud
            response = await server.handle_request(request)
            
            # Enviar respuesta a stdout
            print(json.dumps(response.model_dump(exclude_none=True)))
            sys.stdout.flush()
            
        except KeyboardInterrupt:
            break
        except Exception as e:
            print(f"Error: {e}", file=sys.stderr)
            break

if __name__ == "__main__":
    asyncio.run(main())
'''
    
    print("💻 Código del Bucle Principal en Producción:")
    print("=" * 50)
    print(production_code)

# Ejecutar simulación del bucle principal
print("🎮 Ejecutando simulación del servidor...")
result = await main_server_loop()
print(f"\\n✅ {result}")

# Mostrar código de producción
print("\\n" + "="*60)
show_production_main_loop()

## 10. Probar el Servidor MCP Completo

¡Hora de probar nuestro servidor MCP completo! Realizaremos una **demo integral** que muestra todas las características funcionando juntas.

### 🎯 Objetivos de la Demo:
1. **Verificar integración completa** - Todos los componentes trabajando juntos
2. **Validar protocolo MCP** - Comunicación JSON-RPC 2.0 correcta
3. **Demostrar casos de uso reales** - Escenarios que un cliente real ejecutaría
4. **Mostrar manejo de errores** - Respuestas robustas a entradas inválidas

### 📋 Plan de Pruebas Completas:
1. ✅ **Handshake inicial** (initialize)
2. 🛠️ **Descubrimiento de herramientas** (tools/list)
3. 🔍 **Obtener todos los Pokémon**
4. 🏷️ **Filtrar por tipo específico**
5. 📋 **Detalles de Pokémon individual**
6. ❌ **Casos de error** (herramienta inexistente, parámetros inválidos)
7. 📊 **Análisis de rendimiento**

Esta demo simula cómo un cliente MCP real (como Claude o VS Code) interactuaría con nuestro servidor.

In [ ]:
# Demo completa del servidor MCP Pokemon
import time

async def comprehensive_mcp_demo():
    """Demo completa que simula un cliente MCP real interactuando con nuestro servidor"""
    
    print("🎮 DEMO COMPLETA: Servidor MCP Pokémon")
    print("=" * 60)
    print("🤖 Simulando cliente MCP conectándose al servidor...")
    print()
    
    server = MCPServer()
    start_time = time.time()
    
    # === FASE 1: HANDSHAKE INICIAL ===
    print("🤝 FASE 1: Handshake Inicial")
    print("-" * 30)
    
    init_request = MCPRequest(
        id="init-001",
        method="initialize",
        params={
            "protocolVersion": "2024-11-05",
            "capabilities": {"sampling": {}},
            "clientInfo": {
                "name": "Pokemon-Client-Demo",
                "version": "1.0.0"
            }
        }
    )
    
    response = await server.handle_request(init_request)
    if response.result:
        print("✅ Conexión establecida exitosamente")
        print(f"   🏷️ Servidor: {response.result['serverInfo']['name']}")
        print(f"   📅 Protocolo: {response.result['protocolVersion']}")
        print(f"   ⚙️ Capacidades: {list(response.result['capabilities'].keys())}")
    else:
        print(f"❌ Error en handshake: {response.error}")
        return
    
    # === FASE 2: DESCUBRIMIENTO DE HERRAMIENTAS ===
    print("\\n🔍 FASE 2: Descubrimiento de Herramientas")
    print("-" * 40)
    
    tools_request = MCPRequest(id="tools-001", method="tools/list")
    response = await server.handle_request(tools_request)
    
    if response.result:
        tools = response.result["tools"]
        print(f"✅ Descubiertas {len(tools)} herramientas:")
        for i, tool in enumerate(tools, 1):
            print(f"   {i}. {tool['name']}")
            print(f"      📝 {tool['description']}")
            required_params = tool['inputSchema'].get('required', [])
            if required_params:
                print(f"      🔧 Parámetros requeridos: {', '.join(required_params)}")
            else:
                print(f"      🔧 Sin parámetros requeridos")
    else:
        print(f"❌ Error descubriendo herramientas: {response.error}")
        return
    
    # === FASE 3: EXPLORACIÓN DE DATOS ===
    print("\\n📊 FASE 3: Exploración de Datos")
    print("-" * 35)
    
    # 3.1 Obtener inventario completo
    print("\\n📋 Obteniendo inventario completo...")
    all_request = MCPRequest(
        id="get-all-001",
        method="tools/call",
        params={"name": "get_all_pokemon", "arguments": {}}
    )
    response = await server.handle_request(all_request)
    
    if response.result:
        content = response.result["content"][0]["text"]
        # Extraer estadísticas básicas
        try:
            json_start = content.find('{')
            data = json.loads(content[json_start:])
            total = data['total_count']
            print(f"✅ Inventario cargado: {total} Pokémon disponibles")
            
            # Mostrar muestra
            sample_pokemon = data['pokemon'][:3]
            print("   🎯 Muestra (primeros 3):")
            for p in sample_pokemon:
                print(f"      • {p['name']} ({p['type']}) - {p['weight']}kg")
        except:
            print("✅ Datos obtenidos (formato complejo)")
    else:
        print(f"❌ Error obteniendo inventario: {response.error}")
    
    # 3.2 Análisis por categorías
    print("\\n🏷️ Analizando por categorías...")
    for poke_type in ["Electric", "Fire", "Water", "Psychic"]:
        type_request = MCPRequest(
            id=f"type-{poke_type}",
            method="tools/call",
            params={"name": "get_pokemon_by_type", "arguments": {"pokemon_type": poke_type}}
        )
        response = await server.handle_request(type_request)
        
        if response.result:
            content = response.result["content"][0]["text"]
            if "Encontrados" in content:
                # Extraer número de resultados
                lines = content.split('\\n')
                count_info = lines[0]
                print(f"   ⚡ {poke_type}: {count_info.split('Encontrados ')[1].split(' ')[0]} Pokémon")
            else:
                print(f"   ⚡ {poke_type}: 0 Pokémon")
    
    # === FASE 4: CONSULTAS ESPECÍFICAS ===
    print("\\n🎯 FASE 4: Consultas Específicas")
    print("-" * 35)
    
    # 4.1 Detalles de Pokémon popular
    print("\\n⭐ Consultando detalles de Pikachu...")
    detail_request = MCPRequest(
        id="detail-pikachu",
        method="tools/call",
        params={"name": "get_pokemon_details", "arguments": {"identifier": "Pikachu"}}
    )
    response = await server.handle_request(detail_request)
    
    if response.result:
        content = response.result["content"][0]["text"]
        try:
            json_start = content.find('{')
            pikachu_data = json.loads(content[json_start:])
            print("✅ Detalles obtenidos:")
            print(f"   📊 IMC: {pikachu_data['details']['bmi']}")
            print(f"   🏷️ Tipo: {pikachu_data['type']}")
            print(f"   ⚖️ Peso: {pikachu_data['weight']} kg")
            print(f"   📏 Altura: {pikachu_data['height']} m")
        except:
            print("✅ Detalles obtenidos (procesamiento complejo)")
    
    # 4.2 Búsqueda por ID
    print("\\n🔢 Consultando Pokémon por ID (2)...")
    id_request = MCPRequest(
        id="detail-id2",
        method="tools/call",
        params={"name": "get_pokemon_details", "arguments": {"identifier": "2"}}
    )
    response = await server.handle_request(id_request)
    
    if response.result:
        content = response.result["content"][0]["text"]
        try:
            json_start = content.find('{')
            pokemon_data = json.loads(content[json_start:])
            print(f"✅ ID 2 corresponde a: {pokemon_data['name']} ({pokemon_data['type']})")
        except:
            print("✅ Consulta por ID exitosa")
    
    # === FASE 5: MANEJO DE ERRORES ===
    print("\\n🚨 FASE 5: Pruebas de Robustez")
    print("-" * 35)
    
    # 5.1 Herramienta inexistente
    print("\\n❌ Probando herramienta inexistente...")
    invalid_tool_request = MCPRequest(
        id="invalid-tool",
        method="tools/call",
        params={"name": "invalid_tool", "arguments": {}}
    )
    response = await server.handle_request(invalid_tool_request)
    
    if response.error:
        print(f"✅ Error manejado correctamente: {response.error['message']}")
    else:
        print("❌ Error no detectado")
    
    # 5.2 Pokémon inexistente
    print("\\n🔍 Probando búsqueda de Pokémon inexistente...")
    missing_pokemon_request = MCPRequest(
        id="missing-pokemon",
        method="tools/call",
        params={"name": "get_pokemon_details", "arguments": {"identifier": "Inexistente"}}
    )
    response = await server.handle_request(missing_pokemon_request)
    
    if response.result:
        content = response.result["content"][0]["text"]
        if "no encontrado" in content:
            print("✅ Búsqueda fallida manejada correctamente")
        else:
            print("❌ Debería haber fallado")
    
    # 5.3 Tipo inexistente
    print("\\n🏷️ Probando filtro por tipo inexistente...")
    invalid_type_request = MCPRequest(
        id="invalid-type",
        method="tools/call",
        params={"name": "get_pokemon_by_type", "arguments": {"pokemon_type": "Inexistente"}}
    )
    response = await server.handle_request(invalid_type_request)
    
    if response.result:
        content = response.result["content"][0]["text"]
        if "No se encontraron" in content or "Tipos disponibles" in content:
            print("✅ Tipo inexistente manejado correctamente")
    
    # === ESTADÍSTICAS FINALES ===
    end_time = time.time()
    print("\\n📈 ESTADÍSTICAS DE LA DEMO")
    print("-" * 30)
    print(f"⏱️ Tiempo total: {end_time - start_time:.2f} segundos")
    print(f"📊 Total de solicitudes procesadas: 10+")
    print(f"✅ Todas las fases completadas exitosamente")
    
    print("\\n🎉 DEMO COMPLETADA")
    print("=" * 60)
    print("🏆 El servidor MCP Pokémon está funcionando correctamente!")
    print("🚀 Listo para integración con clientes MCP reales")
    print("\\n💡 Próximos pasos:")
    print("   1. Ejecutar el archivo pokemon_mcp_server.py")
    print("   2. Conectar desde un cliente MCP (Claude, VS Code, etc.)")
    print("   3. Disfrutar de la gestión de datos Pokémon! 🎮")

# Ejecutar la demo completa
await comprehensive_mcp_demo()

## 🎯 Conclusiones y Próximos Pasos

¡Felicidades! Has completado la construcción de un servidor MCP completo para gestión de datos Pokémon. 

### 🏆 Lo que has aprendido:

#### 🔧 Aspectos Técnicos
- **Protocolo MCP**: Implementación completa del Model Context Protocol
- **JSON-RPC 2.0**: Comunicación robusta cliente-servidor
- **Pydantic**: Modelado y validación de datos profesional
- **Python Asyncio**: Programación asíncrona para servidores

#### 🛠️ Arquitectura de Software
- **Separación de responsabilidades**: Modelos, servidor, y herramientas
- **Manejo de errores**: Códigos de error estándar JSON-RPC
- **Extensibilidad**: Fácil adición de nuevas herramientas
- **Testing**: Pruebas comprehensivas de funcionalidad

### 🚀 Próximos Pasos Sugeridos

#### 1. **Extensiones del Servidor**
```python
# Nuevas herramientas que podrías implementar:
- search_pokemon_by_stat (buscar por peso/altura)
- get_pokemon_evolution_chain
- calculate_team_stats
- export_pokemon_data
```

#### 2. **Mejoras de Datos**
- Conectar a PokeAPI real para datos dinámicos
- Agregar más campos (estadísticas de combate, evoluciones)
- Implementar base de datos persistente

#### 3. **Características Avanzadas**
- Autenticación y autorización
- Rate limiting para uso en producción
- Logging y monitoreo
- Configuración via variables de entorno

#### 4. **Integración con Clientes**
- Usar con Claude Desktop
- Crear extensión de VS Code
- Integrar con aplicaciones web

### 📚 Recursos de Aprendizaje

- **[Especificación MCP](https://modelcontextprotocol.io/)**: Documentación oficial
- **[Pydantic Docs](https://docs.pydantic.dev/)**: Modelado de datos avanzado
- **[JSON-RPC 2.0](https://www.jsonrpc.org/specification)**: Estándar de comunicación
- **[AsyncIO Tutorial](https://docs.python.org/3/library/asyncio.html)**: Programación asíncrona

### 🎮 ¡Tu Servidor Está Listo!

Tu servidor MCP Pokémon es completamente funcional y listo para usar. Los archivos del proyecto incluyen:

- `pokemon_mcp_server.py` - Servidor principal
- `demo.py` - Demostración interactiva  
- `test_mcp_server.py` - Suite de pruebas
- `README.md` - Documentación completa

¡Ahora puedes crear tus propios servidores MCP para cualquier dominio de datos que te interese! 🌟